In [1]:
# ===============================
# DATA PREPARATION AND PREPROCESSING ACS WILDA26_01B Team 5
# ===============================

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [3]:
# 1. Load dataset
df = pd.read_csv("Dataset_ATS_v2.csv")

In [4]:
# 2. Display basic information
print("Shape of dataset:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

Shape of dataset: (7043, 10)

First 5 rows:
   gender  SeniorCitizen Dependents  tenure PhoneService MultipleLines  \
0  Female              0         No       1           No            No   
1    Male              0         No      41          Yes            No   
2  Female              0        Yes      52          Yes            No   
3  Female              0         No       1          Yes            No   
4    Male              0         No      67          Yes            No   

  InternetService        Contract  MonthlyCharges Churn  
0             DSL  Month-to-month              25   Yes  
1             DSL        One year              25    No  
2             DSL  Month-to-month              19    No  
3             DSL        One year              76   Yes  
4     Fiber optic  Month-to-month              51    No  

Column names:
['gender', 'SeniorCitizen', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'Contract', 'MonthlyCharges', 'Churn']

Data

In [5]:
# 3. Standardise column names
df.columns = df.columns.str.strip()

In [6]:
# 4. Remove extra spaces from categorical values
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

In [7]:
# 5. Check missing values
print("\nMissing values before handling:")
print(df.isnull().sum())


Missing values before handling:
gender             0
SeniorCitizen      0
Dependents         0
tenure             0
PhoneService       0
MultipleLines      0
InternetService    0
Contract           0
MonthlyCharges     0
Churn              0
dtype: int64


In [8]:
# 6. Check duplicate records
print("\nNumber of duplicate rows:", df.duplicated().sum())


Number of duplicate rows: 302


In [9]:
# Remove duplicates
df = df.drop_duplicates()

In [10]:
# 7. Separate features and target
# Churn is excluded for clustering because clustering is unsupervised
X = df.drop(columns=["Churn"], errors="ignore")

In [11]:
# 8. Identify numeric and categorical columns
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

In [12]:
print("\nNumeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)


Numeric columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges']
Categorical columns: ['gender', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'Contract']


In [13]:
# 9. Split dataset into training and testing sets
X_train, X_test = train_test_split(
    X,
    test_size=0.2,
    random_state=42
)

print("\nTraining set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)


Training set shape: (5392, 9)
Testing set shape: (1349, 9)


In [14]:
# 10. Create preprocessing pipelines

# Numeric pipeline:
# - fill missing numeric values with median
# - apply standard scaling
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [15]:
# Categorical pipeline:
# - fill missing categorical values with most frequent value
# - encode categorical variables using one-hot encoding
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

In [16]:
# 11. Combine preprocessing steps
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

In [17]:
# 12. Apply preprocessing
# Fit on training set, transform both training and testing sets
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [18]:
# 13. Convert processed data to DataFrames
encoded_cat_cols = preprocessor.named_transformers_["cat"]["encoder"].get_feature_names_out(categorical_cols)
all_feature_names = numeric_cols + list(encoded_cat_cols)

X_train_df = pd.DataFrame(
    X_train_processed.toarray() if hasattr(X_train_processed, "toarray") else X_train_processed,
    columns=all_feature_names
)

X_test_df = pd.DataFrame(
    X_test_processed.toarray() if hasattr(X_test_processed, "toarray") else X_test_processed,
    columns=all_feature_names
)

print("\nProcessed training data preview:")
print(X_train_df.head())

print("\nProcessed testing data preview:")
print(X_test_df.head())


Processed training data preview:
   SeniorCitizen    tenure  MonthlyCharges  gender_Male  Dependents_Yes  \
0      -0.451189  1.479092        1.115583          1.0             1.0   
1      -0.451189 -1.274783        0.173486          0.0             0.0   
2      -0.451189 -1.069270       -1.071429          1.0             0.0   
3      -0.451189 -1.110373        1.485693          0.0             0.0   
4      -0.451189 -0.945962       -0.869551          0.0             1.0   

   PhoneService_Yes  MultipleLines_Yes  InternetService_Fiber optic  \
0               1.0                0.0                          0.0   
1               1.0                0.0                          0.0   
2               0.0                0.0                          0.0   
3               1.0                0.0                          0.0   
4               0.0                0.0                          0.0   

   Contract_One year  Contract_Two year  
0                0.0                0.0  
1   

In [19]:
# 14. Data integrity check after preprocessing
print("\nMissing values in processed training set:", X_train_df.isnull().sum().sum())
print("Missing values in processed testing set:", X_test_df.isnull().sum().sum())


Missing values in processed training set: 0
Missing values in processed testing set: 0


In [20]:
# 15. Save cleaned/preprocessed datasets
X_train_df.to_csv("train_preprocessed.csv", index=False)
X_test_df.to_csv("test_preprocessed.csv", index=False)

print("\nPreprocessed datasets saved successfully:")
print("- train_preprocessed.csv")
print("- test_preprocessed.csv")


Preprocessed datasets saved successfully:
- train_preprocessed.csv
- test_preprocessed.csv


In [21]:
# 16. Save the full preprocessed dataset

X_full_df = pd.concat([X_train_df, X_test_df], axis=0)

# Optional: reset index for clean ordering
X_full_df = X_full_df.reset_index(drop=True)

X_full_df.to_csv("full_preprocessed_dataset.csv", index=False)

print("- full_preprocessed_dataset.csv")

- full_preprocessed_dataset.csv
